# 04_dataset_builder: Build the Fine-Tuning Dataset

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/04_dataset_builder.ipynb)

**Session:** Day 2, S10 — Lab: build the dataset
**Expected runtime:** 40 minutes of work on Colab free-tier CPU (or local). The code itself runs in under a minute; the time is yours, for three TODOs and for reading what the checks find.
**Needs:** no API key, no GPU, no Ollama. Reads `corpus/tickets/tickets_raw.jsonl`, `data/finetune/ticket_labels.jsonl`, `data/finetune/ticket_schema.json`, `data/finetune/train.jsonl`, `data/finetune/val.jsonl`, `data/eval/heldout_20.jsonl`, `scripts/quality_checks.py`.
**A correct result looks like:** the final cell prints `DATASET READY` with **373** clean train rows, **72** clean validation rows, **20** held-out rows with **0** overlap, after your checks found **20** near-duplicates, **5** leaked rows and **10** schema violations. `scripts/quality_checks.py` reports three `FAIL`s on the inherited files and none on your cleaned ones. `train_clean.jsonl` and `val_clean.jsonl` are saved in your checkpoint folder (Google Drive on Colab) for the fine-tuning lab.

> All data in this lab is synthetic. No real OQ material anywhere.

---
**The plan.** Part A builds a dataset from raw tickets: load → inspect → build pairs (**TODO 1**) → hold out the exam → split → inspect the splits. Part B is the part that matters on a real project: you inherit a dataset somebody else built and find out what is wrong with it (**TODO 2**, **TODO 3**) before a GPU ever sees it.

**If the runtime disconnects:** reconnect and *Run all*. Everything here recomputes in seconds, your TODO answers live in the notebook itself, and the files that later labs need are already on Drive.

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** each notebook installs only what it needs, with an exact pin that matches `requirements.txt`. This lab needs one package beyond the standard library and pandas: `jsonschema`, to check records against the ticket schema. Colab already ships this exact version, so the install is a two-second no-op there; it is pinned anyway so a Colab image change cannot silently change the lab.

In [ ]:
# Pinned installs — versions match requirements.txt. Colab only;
# local machines installed requirements.txt during setup.
if IN_COLAB:
    %pip install -q jsonschema==4.26.0
print("Install cell done.")

---
## Part A — build a dataset from raw tickets

**Why this cell:** a fine-tuning dataset is built from two things that are kept apart on purpose. The **raw tickets** are what users typed. The **labels** are the structured record a good triage agent would produce for each one. They live in different files, joined by `ticket_id`, so that the retrieval labs on Day 3 can read the tickets without ever seeing the answers.

In [ ]:
import json

import pandas as pd

import dataset_utils
import utils

tickets_path = REPO_ROOT / "corpus" / "tickets" / "tickets_raw.jsonl"
labels_path = REPO_ROOT / "data" / "finetune" / "ticket_labels.jsonl"

tickets = dataset_utils.load_jsonl(tickets_path)
labels = dataset_utils.load_jsonl(labels_path)

print(f"{len(tickets)} raw tickets, {len(labels)} label rows")
print()
print("The first raw ticket:")
print(json.dumps(tickets[0], indent=2))

**Why this cell:** look at the input before you build anything on it. Real tickets are messy, and these are messy on purpose: typos, no punctuation, one-liners, walls of text. If a model only ever trains on tidy paragraphs, it fails on the first real ticket. Sorting by length is the quickest way to see both extremes.

In [ ]:
def word_count(ticket):
    return len(ticket["body"].split())

tickets_by_length = sorted(tickets, key=word_count)
shortest_ticket = tickets_by_length[0]
middle_ticket = tickets_by_length[len(tickets_by_length) // 2]
longest_ticket = tickets_by_length[-1]

print(f"shortest: {word_count(shortest_ticket)} words | "
      f"middle: {word_count(middle_ticket)} words | "
      f"longest: {word_count(longest_ticket)} words")
print()
print("SHORTEST:", repr(shortest_ticket["body"]))
print()
print("MIDDLE  :", repr(middle_ticket["body"]))
print()
print("LONGEST (first 300 characters):", repr(longest_ticket["body"][:300]))

**Why this cell:** now the answers. The category mix is **imbalanced, and we leave it that way**. Password and access tickets dominate a real service desk; ERP and telecom are rare. A balanced dataset would make the tuned model look better than it will be in production, and would hide the rare-class problem that you want to see *today*, not after go-live.

In [ ]:
categories = [label["record"]["category"] for label in labels]
category_counts = pd.Series(categories).value_counts()

print(category_counts.to_string())
print()
imbalance = category_counts.max() / category_counts.min()
print(f"The most common category is {imbalance:.1f}x the rarest.")
print()
print("One label row:")
print(json.dumps(labels[0], indent=2))

**Why this cell:** join each ticket to its label so that one Python object — an *example* — holds the input text, the target record, and `meta`. `meta` is the generator's bookkeeping (which scenario, which writing style). It is for **your** error analysis later. It is never shown to a model and never a training target.

In [ ]:
examples = dataset_utils.join_tickets_and_labels(tickets, labels)

first_example = examples[0]
print(f"{len(examples)} examples. Each has keys: {list(first_example)}")
print()
print("ticket_id:", first_example["ticket"]["ticket_id"])
print("record   :", first_example["record"])
print("meta     :", first_example["meta"])

**Why this cell (TODO 1):** this is the heart of the builder. A fine-tuning *pair* is one conversation with three messages:

| role | content |
|---|---|
| `system` | the instructions — always `dataset_utils.SYSTEM_PROMPT`, the same text at training time and at inference time |
| `user` | the ticket text: subject + body |
| `assistant` | the completion: the record as a JSON string, **and nothing else** |

No "Here is the JSON:", no markdown fence, no trailing comment in the completion. The model learns exactly what you show it: show it chatter and it will produce chatter, and the parser downstream breaks.

Fill in the two gaps. The surrounding code shows you the shape.

In [ ]:
def build_pair(ticket, record):
    user_text = dataset_utils.format_ticket_text(ticket)

    # ── TODO 1 ─────────────────────────────────────────────────────────
    # completion_text: the record as a JSON *string*.
    #   Hint: json.dumps(record, ensure_ascii=False)
    # messages: a list of three dicts, each {"role": ..., "content": ...},
    #   in the order system, user, assistant.
    #   Hint: the system content is dataset_utils.SYSTEM_PROMPT
    completion_text = ...  # <- replace the ... with your code
    messages = ...         # <- replace the ... with your code
    # ───────────────────────────────────────────────────────────────────

    assert completion_text is not ... and messages is not ..., "TODO 1 is not filled in yet"
    return {"ticket_id": ticket["ticket_id"], "messages": messages}


print("build_pair is defined. The next cell checks it.")

**Why this cell:** check your pair against the reference implementation before building 500 of them. If the assertion fails, the message tells you which of the three messages differs.

In [ ]:
my_pair = build_pair(first_example["ticket"], first_example["record"])
reference_pair = dataset_utils.build_pair(first_example["ticket"], first_example["record"])

for position, role in enumerate(["system", "user", "assistant"]):
    mine = my_pair["messages"][position]
    expected = reference_pair["messages"][position]
    assert mine == expected, f"The {role} message differs.\n  yours   : {mine}\n  expected: {expected}"

print("build_pair matches the reference. One pair, as it will appear in train.jsonl:")
print()
print("SYSTEM   :", my_pair["messages"][0]["content"][:140], "...")
print()
print("USER     :", my_pair["messages"][1]["content"])
print()
print("ASSISTANT:", my_pair["messages"][2]["content"])

**Why role/content messages, and not one formatted string?** Every model family wraps a conversation in its own special tokens — its *chat template*. Llama marks turns one way, Qwen another, Gemma another. If we wrote one model's tokens into the data, the data would fit only that model.

So the dataset stays model-neutral, and the template is applied as late as possible, by the tool that owns it:

- **at training time** the tokenizer applies the template (`tokenizer.apply_chat_template`; the trainer does it for you when a dataset has a `messages` column);
- **at inference time** Ollama applies the *same* template when you call `/v1/chat/completions` with the same three roles.

The single most common fine-tuning failure is a mismatch between those two. Keeping one `messages` format end to end is how you avoid it.

**Why this cell:** before splitting anything, lock away the exam. The **held-out 20** are scored in S12 this afternoon (base vs tuned) and again on Day 4 (base vs tuned vs retrieval), so they must never be seen in training, and they must be the *same* 20 for everyone in the room.

They are **stratified, not random**. Twenty random tickets from an imbalanced corpus would probably contain no ERP ticket and no critical one. `select_heldout` guarantees every category, a critical ticket, an enterprise-wide one, and several where urgency is honestly arguable. It also refuses any ticket that has a lookalike elsewhere in the corpus — an exam question with a twin in the textbook is not an exam question.

In [ ]:
SEED = 42

heldout_examples = dataset_utils.select_heldout(examples, seed=SEED)
heldout_ids = [example["ticket"]["ticket_id"] for example in heldout_examples]

committed_heldout = dataset_utils.load_jsonl(REPO_ROOT / "data" / "eval" / "heldout_20.jsonl")
committed_ids = [pair["ticket_id"] for pair in committed_heldout]

assert heldout_ids == committed_ids, "Your held-out 20 differ from data/eval/heldout_20.jsonl"
print(f"{len(heldout_ids)} held-out tickets, identical to data/eval/heldout_20.jsonl")

**Why this cell:** read the exam paper. Look at the `features` column: `tone_urgent` is a user shouting URGENT with no stated business effect (the label follows the *effect*, so urgency stays low or medium); `two_problems` is a ticket raising two unrelated issues (the label describes the *first*). Those rows are where a reasonable colleague could disagree with the label — which is the rubric conversation in S12.

In [ ]:
heldout_rows = []
for example in heldout_examples:
    row = {
        "ticket_id": example["ticket"]["ticket_id"],
        "category": example["record"]["category"],
        "urgency": example["record"]["urgency"],
        "impact": example["record"]["impact"],
        "features": ", ".join(example["meta"]["features"]),
    }
    heldout_rows.append(row)

heldout_table = pd.DataFrame(heldout_rows)
print(heldout_table.to_string(index=False))

**Why this cell:** split what is left into **train** (the model learns from it) and **validation** (watched *during* training to spot overfitting). The split is stratified by category so both keep the real mix. We take 400 + 80; the remaining tickets stay unused — a reserve you can draw on when you extend the builder.

In [ ]:
heldout_id_set = set(heldout_ids)

pool = []
for example in examples:
    if example["ticket"]["ticket_id"] not in heldout_id_set:
        pool.append(example)

train_examples, val_examples, unused_examples = dataset_utils.split_train_val(
    pool, train_size=400, val_size=80, seed=SEED
)

print(f"pool after hold-out : {len(pool)}")
print(f"train               : {len(train_examples)}")
print(f"val                 : {len(val_examples)}")
print(f"unused reserve      : {len(unused_examples)}")

**Why this cell:** never trust a split you have not looked at. Each column below should tell the same story as the corpus: access on top, ERP and telecom at the bottom. The held-out column is deliberately flatter — it over-samples the rare classes so that a per-category score is never computed on zero tickets.

In [ ]:
def category_percent(example_list):
    example_categories = [example["record"]["category"] for example in example_list]
    shares = pd.Series(example_categories).value_counts(normalize=True)
    return (shares * 100).round(1)

split_table = pd.DataFrame({
    "corpus %": category_percent(examples),
    "train %": category_percent(train_examples),
    "val %": category_percent(val_examples),
    "heldout %": category_percent(heldout_examples),
})
print(split_table.to_string())

**Why this cell (milestone 1):** turn every example into a pair with **your** `build_pair`, prove the exam is sealed (zero overlap with train and validation), and save. On Colab `CHECKPOINT_DIR` is on Google Drive, so these files survive a runtime disconnect.

In [ ]:
my_train_pairs = [build_pair(example["ticket"], example["record"]) for example in train_examples]
my_val_pairs = [build_pair(example["ticket"], example["record"]) for example in val_examples]
my_heldout_pairs = [build_pair(example["ticket"], example["record"]) for example in heldout_examples]

my_train_val_ids = {pair["ticket_id"] for pair in my_train_pairs + my_val_pairs}
overlap = heldout_id_set & my_train_val_ids
assert overlap == set(), f"Held-out tickets leaked into train/val: {sorted(overlap)}"
print(f"held-out overlap with train + val: {len(overlap)}  (must be 0)")

dataset_dir = CHECKPOINT_DIR / "04_dataset"
for name, pairs in [("my_train", my_train_pairs), ("my_val", my_val_pairs)]:
    saved_path = dataset_utils.write_jsonl(dataset_dir / f"{name}.jsonl", pairs)
    print(f"checkpoint saved: {saved_path}  ({len(pairs)} rows)")

---
## Part B — the dataset you inherit

Your split is clean: you built it a minute ago, from source, and you watched every step.

That is rarely the dataset you are handed. The repo ships `data/finetune/train.jsonl` and `val.jsonl`. They came out of this same pipeline — and then had a life. A second export of resubmitted tickets was merged in. A few rows were copied across while someone rebalanced validation. Some labels were "fixed" by hand in a text editor.

Nothing crashes. Training would run happily on this file. **Find what is wrong with it before you spend GPU time on it.**

**Why this cell:** load the inherited files and do the cheapest check there is — count the values of an enum field. Read the output closely: a field that can only hold four values should not show six.

In [ ]:
train_pairs = dataset_utils.load_jsonl(REPO_ROOT / "data" / "finetune" / "train.jsonl")
val_pairs = dataset_utils.load_jsonl(REPO_ROOT / "data" / "finetune" / "val.jsonl")
print(f"inherited train: {len(train_pairs)} rows | inherited val: {len(val_pairs)} rows")

for field in ["category", "urgency", "impact"]:
    print()
    print(f"train, values of '{field}':")
    field_counts = dataset_utils.count_record_field(train_pairs, field)
    for value, count in field_counts.items():
        print(f"  {count:>4}  {value}")

**Why this cell (TODO 2) — validation leakage:** if a row sits in both train and validation, the model is graded on an answer it has already been shown. The validation score goes up, and it is a lie: it no longer tells you anything about tickets the model has not seen. The check is one line of set arithmetic.

In [ ]:
train_ids = {pair["ticket_id"] for pair in train_pairs}
val_ids = {pair["ticket_id"] for pair in val_pairs}

# ── TODO 2 ─────────────────────────────────────────────────────────
# leaked_ids: the ticket_ids that are in BOTH sets.
#   Hint: for two sets a and b, the intersection is  a & b
leaked_ids = ...  # <- replace the ... with your code
# ───────────────────────────────────────────────────────────────────

assert leaked_ids is not ..., "TODO 2 is not filled in yet"
print(f"{len(leaked_ids)} ticket_ids appear in BOTH train and val:")
print(sorted(leaked_ids))

**Why this cell:** a check you have only ever seen say "problem found" is a check you cannot trust. Run the same logic on **your own** split from Part A as a control. It should find nothing. (`find_leakage` is the reference version of what you just wrote.)

In [ ]:
reference_leaked_ids = dataset_utils.find_leakage(train_pairs, val_pairs)
my_split_leaked_ids = dataset_utils.find_leakage(my_train_pairs, my_val_pairs)

print(f"inherited dataset, your check : {len(leaked_ids)} leaked")
print(f"inherited dataset, reference  : {len(reference_leaked_ids)} leaked")
print(f"your own split (the control)  : {len(my_split_leaked_ids)} leaked")
assert sorted(leaked_ids) == reference_leaked_ids, "Your check and the reference disagree"


**Why this cell (TODO 3) — schema violations:** the whole point of this fine-tune is *strict JSON that matches the schema*. A training row whose completion breaks the schema teaches the model to break the schema. `jsonschema` does the checking; you parse the completion and ask the validator for its errors.

In [ ]:
from jsonschema import Draft202012Validator

schema = dataset_utils.load_schema(REPO_ROOT / "data" / "finetune" / "ticket_schema.json")
validator = Draft202012Validator(schema)

violations = []
for pair in train_pairs + val_pairs:
    completion_text = dataset_utils.pair_completion_text(pair)

    # ── TODO 3 ─────────────────────────────────────────────────────────
    # record: the completion parsed from a JSON string into a dict.
    #   Hint: json.loads(...)
    # errors: every way the record breaks the schema, as a list.
    #   Hint: list(validator.iter_errors(record))
    record = ...  # <- replace the ... with your code
    errors = ...  # <- replace the ... with your code
    # ───────────────────────────────────────────────────────────────────

    assert record is not ... and errors is not ..., "TODO 3 is not filled in yet"
    for error in errors:
        field = error.path[0] if error.path else "(whole record)"
        violations.append({
            "ticket_id": pair["ticket_id"],
            "field": field,
            "rule": error.validator,
            "problem": error.message,
        })

violating_ids = sorted({violation["ticket_id"] for violation in violations})
print(f"{len(violating_ids)} rows break the schema:")
for violation in violations:
    print(f"  {violation['ticket_id']}  {violation['field']:<17} [{violation['rule']}]  {violation['problem'][:70]}")

**Why this cell — near-duplicates:** users resubmit. The same ticket comes back a day later with a new greeting, two words changed and a **new ticket id** — so the id check you just wrote cannot see it. Duplicates make the model over-weight whatever happens to be repeated, and when one copy lands in validation they leak as well.

This check is pre-written. It compares the *words* of every ticket with every other (`text_similarity`: shared words / all words) and flags pairs at or above **0.8**. In the 600 raw tickets no two different tickets reach 0.8, so anything flagged here is worth a human look.

In [ ]:
near_duplicates = dataset_utils.find_near_duplicates(train_pairs, threshold=0.8)
print(f"{len(near_duplicates)} near-duplicate pairs in inherited train")

control_near_duplicates = dataset_utils.find_near_duplicates(my_train_pairs, threshold=0.8)
print(f"{len(control_near_duplicates)} in your own train split (the control)")
print()
for item in near_duplicates[:5]:
    print(f"  {item['similarity']:.2f}  {item['first']}  ~  {item['second']}")
print("  ...")

**Why this cell:** a similarity score is a claim, not a fact. Read one flagged pair and decide for yourself whether it is the same ticket. Then try `threshold=0.7` in the cell above: one more pair appears. Is it a duplicate, or two people with the same broken headset? At `0.6` there are eight more, and most are plainly different tickets that share a template. Where you put the threshold is a judgement call, and it is *your* call — not the library's.

In [ ]:
train_pairs_by_id = {pair["ticket_id"]: pair for pair in train_pairs}

example_item = near_duplicates[len(near_duplicates) // 2]
first_pair = train_pairs_by_id[example_item["first"]]
second_pair = train_pairs_by_id[example_item["second"]]

print(f"similarity {example_item['similarity']:.2f}")
print()
print(f"--- {first_pair['ticket_id']} ---")
print(dataset_utils.pair_user_text(first_pair))
print()
print(f"--- {second_pair['ticket_id']} ---")
print(dataset_utils.pair_user_text(second_pair))

**Why this cell:** the three checks above are the core of `scripts/quality_checks.py`, the command-line checker. It adds two things you have not looked at yet — **coverage gaps** and **class imbalance** — and it is what you would put in a pipeline, so that nobody has to remember to open a notebook. It runs in about a second, and its **exit code** is `1` when it finds a hard failure: that is what stops a pipeline before GPU time is spent.

Read the first eight lines of the report. Do its three `FAIL` numbers match yours?

In [ ]:
import subprocess

quality_checks_path = REPO_ROOT / "scripts" / "quality_checks.py"
inherited_dir = REPO_ROOT / "data" / "finetune"

command = [sys.executable, str(quality_checks_path), "--dataset", str(inherited_dir)]
completed = subprocess.run(command, capture_output=True, text=True)

print(completed.stdout + completed.stderr)
print(f"exit code: {completed.returncode}   (0 = clean or warnings only, 1 = hard failures, 2 = unreadable input)")

**Why this cell:** found is not fixed. Drop the bad rows:

- a **leaked** row is removed from *validation* — training data is the scarcer resource, keep it there;
- a **schema-violating** row is removed wherever it is — repairing a label by guesswork is how it broke in the first place;
- of each **near-duplicate** pair, keep the first and drop the second.

In [ ]:
duplicate_ids = {item["second"] for item in near_duplicates}
bad_record_ids = set(violating_ids)

train_clean = []
for pair in train_pairs:
    if pair["ticket_id"] in duplicate_ids or pair["ticket_id"] in bad_record_ids:
        continue
    train_clean.append(pair)

val_clean = []
for pair in val_pairs:
    if pair["ticket_id"] in leaked_ids or pair["ticket_id"] in bad_record_ids:
        continue
    val_clean.append(pair)

print(f"train: {len(train_pairs)} -> {len(train_clean)} rows  ({len(train_pairs) - len(train_clean)} dropped)")
print(f"val  : {len(val_pairs)} -> {len(val_clean)} rows  ({len(val_pairs) - len(val_clean)} dropped)")

**Why this cell:** re-run every check on the cleaned data. A cleaning step that is not verified is just another hand edit. All four numbers must be zero — including the one that matters most this afternoon, the overlap with the held-out 20.

In [ ]:
remaining_leaks = dataset_utils.find_leakage(train_clean, val_clean)
remaining_violations = dataset_utils.find_schema_violations(train_clean + val_clean, schema)
remaining_duplicates = dataset_utils.find_near_duplicates(train_clean + val_clean, threshold=0.8)

clean_ids = {pair["ticket_id"] for pair in train_clean + val_clean}
heldout_overlap = heldout_id_set & clean_ids

print(f"leaked rows left          : {len(remaining_leaks)}")
print(f"schema violations left    : {len(remaining_violations)}")
print(f"near-duplicate pairs left : {len(remaining_duplicates)}")
print(f"overlap with held-out 20  : {len(heldout_overlap)}")

assert remaining_leaks == [] and remaining_violations == [] and remaining_duplicates == []
assert heldout_overlap == set()

**Why this cell (milestone 2):** save the cleaned dataset and a one-screen summary of what you found. On Colab this goes to Google Drive, which is where the fine-tuning lab (S11) looks for it after lunch — on a fresh runtime that has never seen this notebook's memory.

In [ ]:
train_clean_path = dataset_utils.write_jsonl(dataset_dir / "train_clean.jsonl", train_clean)
val_clean_path = dataset_utils.write_jsonl(dataset_dir / "val_clean.jsonl", val_clean)
print(f"checkpoint saved: {train_clean_path}  ({len(train_clean)} rows)")
print(f"checkpoint saved: {val_clean_path}  ({len(val_clean)} rows)")

summary = {
    "train_clean_rows": len(train_clean),
    "val_clean_rows": len(val_clean),
    "heldout_rows": len(committed_heldout),
    "heldout_overlap": len(heldout_overlap),
    "near_duplicates_found": len(near_duplicates),
    "leaked_rows_found": len(leaked_ids),
    "schema_violations_found": len(violating_ids),
}
summary_path = utils.save_json(dataset_dir, "04_summary", summary)

**Why this cell:** the same checker, this time imported as a module and pointed at the two files you just saved. Two things to see.

1. The three hard checks now say `PASS`, so a pipeline would let this dataset through.
2. The two warnings are **still there**. Cleaning did not touch the class imbalance, and nothing in this repo will: there is deliberately no `rebalance` function. Access tickets outnumber ERP tickets eight to one because that is what a service desk sees.

**Decide with your group, there is no free option:** keep the real mix and report *per-class* scores instead of one average; collect and label more of the rare tickets; or re-weight the rare classes in training. Keep your answer in mind for S12, where the tuned model gets scored.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import quality_checks

clean_dataset = quality_checks.load_dataset(train_clean_path, val_path=val_clean_path)
clean_results = quality_checks.run_checks(clean_dataset)
quality_checks.print_report(clean_dataset, clean_results)

still_failing = quality_checks.has_hard_failures(clean_dataset, clean_results)
assert not still_failing, "The cleaned files still have hard failures - read the report above"

**Why this cell:** the last cell always prints the result the header promised, so "done" is checkable at a glance. It reads the summary back from the checkpoint folder — if this cell prints, the files are really there.

In [ ]:
saved_summary = utils.load_json(dataset_dir, "04_summary")

expected_summary = {
    "train_clean_rows": 373,
    "val_clean_rows": 72,
    "heldout_rows": 20,
    "heldout_overlap": 0,
    "near_duplicates_found": 20,
    "leaked_rows_found": 5,
    "schema_violations_found": 10,
}

print(f"{'':<26}{'yours':>8}{'expected':>10}")
for name, expected_value in expected_summary.items():
    your_value = saved_summary[name]
    mark = "ok" if your_value == expected_value else "<-- differs"
    print(f"{name:<26}{your_value:>8}{expected_value:>10}  {mark}")

print()
if saved_summary == expected_summary:
    print(f"DATASET READY — environment={'Colab' if IN_COLAB else 'local'}, files in {dataset_dir}")
else:
    print("NOT YET — a number differs from the reference. Re-check the TODO it belongs to.")